### **Change Data Feed (CDF)**

El Change Data Feed (o Registro de Cambios de Datos) es una característica que registra a nivel de fila y columna todos los cambios realizados en una tabla Delta, incluyendo inserciones (`INSERT`), eliminaciones (`DELETE`) y actualizaciones (`UPDATE`).

En lugar de solo registrar la operación (como lo hace el Registro de Transacciones en la carpeta `_delta_log/`), el CDF registra qué filas exactas fueron modificadas y cómo cambiaron.

**Analizando la Evolución**

- **Registro de Transacciones (`_delta_log)`:** Te dice qué archivos Parquet son válidos en cada versión. Te permite hacer Time Travel (consultar el estado de la tabla en el pasado).

- **Change Data Feed (CDF):** Te dice qué filas específicas y columnas cambiaron entre la Versión `N` y la Versión `N+1`. Te permite hacer streaming o procesamiento por lotes solo de los cambios.

**Usos y Beneficios del CDF**

El CDF resuelve el problema de tener que procesar la tabla completa solo para encontrar los cambios.

**1. Replicación y Sincronización**

Es el caso de uso más común. Permite construir pipelines que replican o sincronizan una tabla Delta grande (la Fuente) con otra tabla o sistema de destino (el Sink) de manera eficiente.

- **Ejemplo:** Tienes una tabla Platinum (Agregada) y quieres actualizarla inmediatamente después de que la tabla Gold (Base de Datos Operacional) cambie. En lugar de procesar toda la tabla Gold, solo lees el CDF para ver las 100 filas que cambiaron y las aplicas al Platinum.

**2. Auditoría y Cumplimiento**

Permite auditar con precisión quién, cuándo y cómo cambió una fila específica, lo cual es fundamental para el cumplimiento normativo.

**3. Propagación de Datos Eficiente**

Esencial para la Arquitectura Medallion (Bronze → Silver → Gold) para crear pipelines incrementales que solo propagan los cambios:

- **Sin CDF:** Para actualizar la capa Silver, tendrías que hacer un `MERGE` comparando toda la capa Bronze con la Silver.

- **Con CDF:** Simplemente lees el CDF de la capa Bronze para obtener solo las filas que cambiaron y las aplicas a la capa Silver. Esto reduce drásticamente el tiempo de procesamiento y el costo.

**Habilitación y Uso**

**1. Habilitación**

Para usar el CDF, primero debes habilitarlo en tu tabla Delta. Puedes hacerlo en la creación de la tabla o en una tabla existente:

In [0]:
%sql
-- Habilitar CDF en una tabla existente
ALTER TABLE my_table SET TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
);

In [0]:
%sql
-- Habilitar al crear una tabla
CREATE TABLE nombre_de_la_tabla (
  id INT,
  nombre STRING
)
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
);

%md
**Lectura de Cambios**

Una vez habilitado, puedes leer el feed de cambios como si fuera una tabla normal, especificando el rango de versiones o marcas de tiempo:

In [0]:
%sql
-- Leer los cambios (INSERTs, UPDATEs, DELETEs) entre la versión 3 y la versión 5
SELECT * FROM table_changes('my_table')
WHERE _commit_version BETWEEN 3 AND 5;

-- O leer los cambios desde un punto en el tiempo
SELECT * FROM table_changes('my_table')
WHERE _commit_timestamp > '2025-11-20 10:00:00';

## **Slowly Changing Dimensions (SCD)**

### **SCD Type 0 (Retain Original)**

Los datos nunca cambian. Una vez que se escriben, se quedan así para siempre.

- **Ejemplo:** Fecha de nacimiento o el ID original de un contrato.

### **SCD Type 1 (Overwrite)**

Cuando cambia una fila, el valor anterior es sobreescrito.

- Aunque esta opción requiere almacenamiento mínimo, se pierde el tracking pasado.

We have a `customers` table as our target. The table currently contains two customers:

- `customer_id 1`, Peter 
- `customer_id 2`, Samarth

We have an `updates` table as our source, this contains:

- UPDATEs (Peter has had two updates on his address. One on 5/15 and the other on 5/20, `customer_id 1`)
- DELETEs (Samarth wants to be removed, `customer_id 2`)
- INSERTs (New customer Kostas, `customer_id 3`)

Our goal is to update the `customers` table with the new customer information from the `updates` source table.

![](https://cdn5.dcbstatic.com/files/d/a/databricks_docebosaas_com/1768428000/4A17hB9gCkgWHmQbFFmZ7A/authoring/688/688_full_slide35_1.jpg)

### **SCD Type 2 (Add New Row)**

Preserva el historial completo añadiendo una nueva fila cada vez que un atributo cambia.

Para esto, añadimos columnas de control:

- `is_current`: Un booleano (True/False).

- `start_date` / `end_date`: Para saber el rango de validez.

| `Customer_ID` | `Ciudad`       | `is_current` | `start_date` | `end_date`   |
|-------------|--------------|------------|------------|------------|
| 101         | Mendoza      | False      | 2024-01-01 | 2026-01-07 |
| 101         | Buenos Aires | True       | 2026-01-08 | NULL       |

- Columns indicate indicate **active** and **inactive** rows 

- **Null** values indicate the **current row**

- **Non Null** values indicate **inactive (historic) rows**

![](https://images.squarespace-cdn.com/content/v1/65a7c5c379a13972f7c20d75/b9ce5f75-00c0-4843-9263-c982bd18a123/Managing+Data+Changes+with+SCDs+5.png?format=1500w)

### **Applying Auto CDC on Lakeflow Declarative Pipelines**

In [0]:
%sql
-- Customers (target table with all customers)
CREATE TABLE IF NOT EXISTS associate_data_engineer.default.customers (
  CustomerID INT,
  Name STRING,
  Address STRING,
  ProcessDate DATE
);

-- Updates (source table with updates)
CREATE TABLE IF NOT EXISTS associate_data_engineer.default.updates (
  CustomerID INT,
  Name STRING,
  Address STRING,
  Operation STRING,
  ProcessDate DATE
);

INSERT INTO associate_data_engineer.default.updates VALUES
  (1, 'Peter', '99 Sunny St.', 'UPDATE', '2025-05-15'), -- Peter cambia de dirección
  (1, 'Peter', '123 Main St.', 'UPDATE', '2025-05-20'),  -- Peter cambia de dirección otra vez
  (2, 'Samarth', '22 Front St', 'DELETE', '2025-05-20'), -- Eliminamos a Samarth
  (3, 'Kostas', '100 Athens Dr.', 'NEW', '2025-05-20'); -- Nuevo cliente

INSERT INTO associate_data_engineer.default.customers VALUES
  (1, 'Peter', '1 Blue Rd.', '2025-05-01'),
  (2, 'Samarth', '22 Front St', '2025-05-01');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE associate_data_engineer.default.customers; -- Create a streaming table, then use AUTO CDC to populate it

CREATE FLOW scd_type_1_flow AS 
AUTO CDC INTO associate_data_engineer.default.customers -- Apply updates, inserts and deletes to the target table
  FROM STREAM associate_data_engineer.default.updates -- Source records to determine updates, deletes and inserts
  KEYS (CustomerID) -- The columns that uniquely identify a row in the source and target tables
APPLY AS DELETE WHEN operation = 'DELETE' -- Specifies when a CDC event should be treated as a DELETE rather than UPSERT
SEQUENCE BY ProcessDate -- Speficies the logical order of CDC events in the source data
COLUMNS * EXCEPT (operation) -- Specifies a subset of columns to include in the target
STORED AS SCD TYPE 1; -- Wheter to store records as SCD Type 1 (default) or Type 2

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8990088215985811>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE OR REFRESH STREAMING TABLE associate_data_engineer.default.customers; -- Create a streaming table, then use AUTO CDC to populate it\n\nCREATE FLOW scd_type_1_flow AS \nAUTO CDC INTO associate_data_engineer.default.customers -- Apply updates, inserts and deletes to the target table\n  FROM STREAM associate_data_engineer.default.updates -- Source records to determine updates, deletes and inserts\n  KEYS (CustomerID) -- The columns that uniquely identify a row in the source and target tables\nAPPLY AS DELETE WHEN operation = 'DELETE' -- Specifies when a CDC event should be treated as a DELETE rather than UPSERT\nSEQUENCE BY ProcessDate -- Speficies the logical order of CDC events in the source data\nCOLUMNS * EXCEPT (operation) -- Specifies a su